In [1]:
!pip install pyarrow

## Data loading

In [2]:
dtype = {
    'date': str,  
    'line': str,
    'task': str,
    'stop_seq': 'float64',    
    'stop_name': str,
    'stop_id': str,
    'scheduled_arrival': str,  
    'scheduled_departure': str,
    'actual_arrival': str,
    'actual_departure': str,
    'detection_type': str,
    'delay': 'float64',
    'stop_desc': str,
    'stop_lat': 'float64',
    'stop_lon': 'float64', 
}

In [3]:
import gc
gc.collect()

0

In [4]:
import pandas as pd
df = pd.read_parquet("/teamspace/studios/this_studio/huge_delays_removed_240625.parquet")

In [5]:
df = df.rename(columns={
    'Dzien': 'date',
    'Linia': 'line',
    'Zadanie': 'task',
    'Lp przystanku': 'stop_seq',
    'Przystanek nazwa': 'stop_name',
    'Przystanek numer': 'stop_id',
    'Rozkladowy czas przyjazdu': 'scheduled_arrival',
    'Rozkladowy czas odjazdu': 'scheduled_departure',
    'Rzeczywisty czas przyjazdu': 'actual_arrival',
    'Rzeczywisty czas odjazdu': 'actual_departure',
    'Rodzaj detekcji': 'detection_type',
})

df = df.astype(dtype)

## Data overview

In [6]:
print("Data types:")
print(df.dtypes)

print("First 5 rows:")
df.head()

Data types:
date                                           object
line                                           object
task                                           object
stop_seq                                      float64
stop_name                                      object
stop_id                                        object
scheduled_arrival                              object
scheduled_departure                            object
actual_arrival                                 object
actual_departure                               object
detection_type                                 object
Primary Key                                    object
stop_desc                                      object
stop_lat                                      float64
stop_lon                                      float64
delay                                         float64
scheduled_trip_start    datetime64[us, Europe/Warsaw]
dtype: object
First 5 rows:


,date,line,task,stop_seq,stop_name,stop_id,scheduled_arrival,scheduled_departure,actual_arrival,actual_departure,detection_type,Primary Key,stop_desc,stop_lat,stop_lon,delay,scheduled_trip_start
0,2023-01-01,10,010-01,1.0,Władysława IV,2169,2023-01-01 04:11:00,2023-01-01 04:11:00,2023-01-01 04:10:07,2023-01-01 04:10:07,1,2023-01-01-010-01-1,Władysława IV,54.39869,18.67434,-53.0,2023-01-01 04:11:00+01:00
1,2023-01-01,10,010-01,2.0,Nowy Port Góreckiego,208,2023-01-01 04:12:00,2023-01-01 04:12:00,2023-01-01 04:12:12,2023-01-01 04:12:44,2,2023-01-01-010-01-1,Nowy Port Góreckiego,54.40005,18.67098,12.0,2023-01-01 04:11:00+01:00
2,2023-01-01,10,010-01,3.0,Marynarki Polskiej,2166,2023-01-01 04:14:00,2023-01-01 04:14:00,2023-01-01 04:13:36,2023-01-01 04:14:00,2,2023-01-01-010-01-1,Marynarki Polskiej,54.39837,18.66617,-24.0,2023-01-01 04:11:00+01:00
3,2023-01-01,10,010-01,4.0,Śnieżna,2164,2023-01-01 04:16:00,2023-01-01 04:16:00,2023-01-01 04:14:56,2023-01-01 04:15:39,2,2023-01-01-010-01-1,Śnieżna,54.39228,18.65767,-64.0,2023-01-01 04:11:00+01:00
4,2023-01-01,10,010-01,5.0,Polsat Plus Arena Gdańsk,2162,2023-01-01 04:18:00,2023-01-01 04:18:00,2023-01-01 04:16:35,2023-01-01 04:17:33,2,2023-01-01-010-01-1,Polsat Plus Arena Gdańsk,54.38726,18.64840,-85.0,2023-01-01 04:11:00+01:00


## Date columns to datetime

In [7]:
datetime_cols = ['scheduled_arrival', 'scheduled_departure', 'date']

for col in datetime_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

# Feature enineering

In [8]:
df['departure_year'] = df['scheduled_departure'].dt.year
df['departure_month'] = df['scheduled_departure'].dt.month
df['departure_day'] = df['scheduled_departure'].dt.day
df['departure_decimal_hour'] = (
    df['scheduled_departure'].dt.hour +
    df['scheduled_departure'].dt.minute / 60
)

df['departure_dow'] = df['date'].dt.dayofweek

In [9]:
def assign_time_of_day(hour):
    if pd.isna(hour):
        return 'unknown'
    elif hour <= 5:
        return 'night'
    elif hour <= 11:
        return 'morning'
    elif hour <= 17:
        return 'afternoon'
    elif hour <= 21:
        return 'evening'
    else:
        return 'late_evening'

In [10]:
df['time_of_day'] = df['departure_decimal_hour'].apply(assign_time_of_day)

## Adding lags (?)

# Dropping unnecesary columns

In [11]:
cols_to_drop = ['scheduled_arrival', 'scheduled_departure', 'actual_arrival', 'actual_departure', 'stop_desc', 'stop_lat', \
    'stop_lon', 'task', "date", "scheduled_trip_start", 'detection_type', 'Primary Key', 'stop_id']
df = df.drop(columns=cols_to_drop)


df = df.dropna(subset=['delay'])

In [12]:
df.columns

Index(['line', 'stop_seq', 'stop_name', 'delay', 'departure_year',
       'departure_month', 'departure_day', 'departure_decimal_hour',
       'departure_dow', 'time_of_day'],
      dtype='object')

## W&B

In [13]:
!pip install wandb

In [14]:
import wandb

In [15]:
TOKEN_PATH = "/teamspace/studios/this_studio/W&B_API_TOKEN.txt"

with open(TOKEN_PATH, "r") as f:
    api_key = f.read().strip()

wandb.login(key=api_key)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /teamspace/studios/this_studio/.netrc
wandb: Currently logged in as: s27478 (dsc-pjatk-warsaw) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [16]:
wandb.init(
    project="bus-delay-prediction",
    entity="dsc-pjatk-warsaw",
    name="pycaret"
)

## Pycaret

In [17]:
!pip install pycaret

In [18]:
import pycaret
pycaret.__version__

'3.3.2'

In [19]:
from pycaret.regression import RegressionExperiment
exp = RegressionExperiment()

In [20]:
split_index = int(len(df) * 0.9)

df_train = df.iloc[:split_index].copy()
df_test = df.iloc[split_index:].copy()

In [21]:
exp.setup(data = df_train, 
                target = 'delay',
                experiment_name='pycaret_lightning_ai',
                log_experiment='wandb',
                session_id=42,

                fold=3,
                n_jobs=-1,
          
                normalize = True,
                transformation = True,
                remove_multicollinearity = True,
                multicollinearity_threshold = 0.95,

                fold_strategy='timeseries',
                fold_shuffle=False,
                data_split_shuffle=False)

,Description,Value
0,Session id,42
1,Target,delay
2,Target type,Regression
3,Original data shape,"(67273425, 10)"
4,Transformed data shape,"(67273425, 14)"
5,Transformed train set shape,"(47091397, 14)"
6,Transformed test set shape,"(20182028, 14)"
7,Numeric features,6
8,Categorical features,3
9,Preprocess,True


In [23]:
exp.save_experiment('my_experiment')

In [24]:
best_cpu = exp.compare_models()

,,
,,
Initiated,. . . . . . . . . . . . . . . . . .,13:35:00
Status,. . . . . . . . . . . . . . . . . .,Fitting 3 Folds
Estimator,. . . . . . . . . . . . . . . . . .,Random Forest Regressor


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
lasso,Lasso Regression,98.5408,19876.1580,140.9001,0.1309,1.3673,2.9123,142.9600
llar,Lasso Least Angle Regression,98.5407,19876.1595,140.9001,0.1309,1.3672,2.9123,139.2367
ridge,Ridge Regression,98.5181,19878.3555,140.9118,0.1308,1.3739,2.9027,143.1167
lar,Least Angle Regression,98.5181,19878.3555,140.9118,0.1308,1.3739,2.9027,142.1867
br,Bayesian Ridge,98.5181,19878.3549,140.9118,0.1308,1.3739,2.9027,151.3700
en,Elastic Net,98.7966,20139.9891,141.8225,0.1196,1.3161,2.8545,146.6433
huber,Huber Regressor,96.5072,21271.9755,145.7206,0.0707,1.4687,2.1570,163.8333
omp,Orthogonal Matching Pursuit,103.6711,21359.6080,146.0452,0.0665,1.4317,3.1968,154.9533
knn,K Neighbors Regressor,101.0939,21984.0632,148.2145,0.0377,1.4849,2.9309,689.1400
par,Passive Aggressive Regressor,101.4631,23405.8196,152.8816,-0.0252,1.5420,2.1361,158.2400


Processing:   0%|          | 0/81 [00:00<?, ?it/s]

: 

In [32]:
xgb_model = exp.create_model('xgboost')

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,90.0114,17698.4297,133.0354,0.2035,1.4233,2.3394
1,95.0368,19663.8809,140.2280,0.2132,1.4372,2.3909
2,89.1128,16336.1807,127.8131,0.2363,1.3742,2.7371
Mean,91.3870,17899.4971,133.6922,0.2177,1.4116,2.4891
Std,2.6068,1365.9473,5.0896,0.0138,0.0270,0.1766


In [34]:
exp.save_model(xgb_model, 'xgboost')

Transformation Pipeline and Model Successfully Saved


(Pipeline(memory=Memory(location=None),
          steps=[('numerical_imputer',
                  TransformerWrapper(include=['stop_seq', 'departure_year',
                                              'departure_month', 'departure_day',
                                              'departure_decimal_hour',
                                              'departure_dow'],
                                     transformer=SimpleImputer())),
                 ('categorical_imputer',
                  TransformerWrapper(include=['line', 'stop_name',
                                              'time_of_day'],
                                     transformer=SimpleImputer(strategy='m...
                               feature_types=None, gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=None,
                               max_bin=None, max_cat_threshold=None,
                           